# Run a Single Coupled Example - Wflow -> SFINCS (inland fluvial/pluvial)

Runs one Event-Catalog event through the inland coupling: stage the event's scaled rainfall
forcing for Wflow, replay the selected Wflow domain set, merge routed discharge at the
SFINCS handoff sources (`sfincs_discharge.nc`), stage the selected Austin SFINCS domain run,
force it with both direct rainfall (`sfincs_netampr.nc`) and the Wflow discharge GeoDataset,
run SFINCS, then plot QA and flood diagnostics.

> **Stage Contract**
>
> Requires: a built Event Catalog, local AORC storm-window files, built Wflow submodels +
> SFINCS base model(s) (`build_coupled_model.ipynb`), a Wflow engine (`WFLOW_BIN` /
> `wflow.run.command`) for replay, and a SFINCS engine (`SFINCS_BIN` or docker) for the run.
>
> Produces: `data/wflow/events/<event>/precip.nc`, `temp_pet.nc`, `sfincs_discharge.nc`, a
> production scenario folder under `data/sfincs/scenarios/<event>/<domain>/`, and flood +
> discharge diagnostics.
>
> Next: `05_create_scenarios.ipynb` stages the full Event Catalog against the same base models.


In [ ]:
import os
import json
import shutil
import datetime
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import yaml
import matplotlib.pyplot as plt
from IPython.display import HTML, Video, display

from sfincs_runs.config import load_runtime
from sfincs_runs.build_base import is_built_sfincs_base
from sfincs_runs.scenarios.event_forcing import run_sfincs_model
from sfincs_runs.scenarios import plan_inland_coupled_example, stage_inland_coupled_scenarios
from sfincs_runs.diagnostics import (
    plot_inland_coupled_forcing_qa,
    plot_inland_coupled_postrun_diagnostics,
    plot_inland_flood_animation,
)
from wflow_runs import build_event_meteo_forcing, replay_inland_domain_set
from hydromt_sfincs import SfincsModel

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parents[1]
repo_root = location_root.parents[1]
config, paths = load_runtime(location_root / "config.yaml")
# Wflow domain review policy is workflow policy, not source/model-recipe YAML.
wflow_domain_review_required = True
config["wflow"]["domain_set"]["review_required"] = wflow_domain_review_required
events_root = location_root / config["wflow"].get("events_root", "data/wflow/events")
scenarios_root = location_root / config.get("paths", {}).get("scenarios_root", "data/sfincs/scenarios")
print(f"location: {location_root.name}  |  root: {location_root}")


def env_flag(name, default):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() not in {"0", "false", "no", "off"}


def command_available(command):
    if not command:
        return False
    executable = str(command).split()[0]
    path = Path(executable)
    return path.exists() or shutil.which(executable) is not None


## 1 - Select a catalog event and plan the coupled example

`plan_inland_coupled_example` checks the static/base-model contract: Event Catalog,
Wflow-SFINCS handoff manifest, built Wflow submodel(s), and built SFINCS domain base(s).
Leave `EVENT_ID = None` for the highest-return-period catalog event, or set a specific id
present in the handoff manifest.


In [ ]:
EVENT_ID = None  # None -> highest-RP catalog event; or e.g. "design_0369"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_inland_coupled_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)

plan_summary = pd.Series({
    "status": example_plan.status,
    "event_id": example_plan.event_id,
    "event_reference_time": example_plan.event_reference_time,
    "catalog_path": example_plan.catalog_path,
    "handoff_path": example_plan.handoff_path,
    "wflow_event_dir": example_plan.wflow_event_dir,
    "wflow_discharge_forcing": example_plan.wflow_discharge_forcing,
    "sfincs_scenario_dir": example_plan.sfincs_scenario_dir,
    "sfincs_dry_run_command": example_plan.sfincs_dry_run_command,
}, name="inland_coupled_example_plan")
display(plan_summary)

if example_plan.issues:
    print("Blocking issues (resolve upstream, then rerun):")
    for issue in example_plan.issues:
        print("  -", issue)
if example_plan.status != "ready":
    raise RuntimeError("Coupled example inputs are not ready; resolve the blocking issues above.")

event_id = example_plan.event_id
discharge_nc = events_root / event_id / "sfincs_discharge.nc"
event_precip_nc = events_root / event_id / "precip.nc"


## 2 - Stage Wflow meteo forcing and replay -> `sfincs_discharge.nc`

`build_event_meteo_forcing` writes the per-event Wflow forcing contract: `precip.nc` is the
catalog-selected AORC storm window scaled by `rainfall_scale_factor`, and `temp_pet.nc` is the
neutral De Bruin PET companion required by the current HydroMT-Wflow update config.

`replay_inland_domain_set` then resolves the event window, writes the per-event HydroMT update
config + data catalog, runs `hydromt update wflow_sbm` and the Wflow engine for each selected
submodel, and merges each submodel's `Q` at `gauges_sfincs` into one `sfincs_discharge.nc`
keyed by `sfincs_handoff_id`.

Run-All plans the replay by default. Set
`FLOOD_RM_RUN_WFLOW_REPLAY=1` to execute the Wflow replay, or
`FLOOD_RM_FORCE_WFLOW_REPLAY=1` to rebuild an existing event discharge file.


In [ ]:
BUILD_WFLOW_METEO = env_flag("FLOOD_RM_BUILD_WFLOW_METEO", True)
OVERWRITE_WFLOW_METEO = env_flag("FLOOD_RM_OVERWRITE_WFLOW_METEO", False)
FORCE_WFLOW_REPLAY = env_flag("FLOOD_RM_FORCE_WFLOW_REPLAY", False)

wflow_run_cfg = config.get("wflow", {}).get("run", {}) or {}
wflow_command = wflow_run_cfg.get("command") or os.environ.get(wflow_run_cfg.get("bin_env", "WFLOW_BIN"), "") or "wflow_cli"
wflow_command_available = command_available(wflow_command)
RUN_WFLOW_REPLAY = env_flag("FLOOD_RM_RUN_WFLOW_REPLAY", False)

display(pd.Series({
    "build_wflow_meteo": BUILD_WFLOW_METEO,
    "overwrite_wflow_meteo": OVERWRITE_WFLOW_METEO,
    "wflow_command": wflow_command,
    "wflow_command_available": wflow_command_available,
    "run_wflow_replay": RUN_WFLOW_REPLAY,
    "force_wflow_replay": FORCE_WFLOW_REPLAY,
}, name="wflow_replay_controls"))

if BUILD_WFLOW_METEO:
    meteo_report = build_event_meteo_forcing(
        config,
        location_root,
        event_id,
        catalog_path=example_plan.catalog_path,
        overwrite=OVERWRITE_WFLOW_METEO,
    )
    display(pd.Series(meteo_report, name="wflow_event_meteo_forcing"))

execute_wflow_replay = RUN_WFLOW_REPLAY and (FORCE_WFLOW_REPLAY or not discharge_nc.exists())
replay_report = replay_inland_domain_set(
    config,
    location_root,
    event_id,
    catalog_path=example_plan.catalog_path,
    execute=execute_wflow_replay,
)
display(replay_report[["submodel_id", "status", "window_start", "window_end", "hydromt_runner_status", "run_output_dir", "sfincs_discharge_forcing"]])

if RUN_WFLOW_REPLAY and not execute_wflow_replay:
    print("Reusing existing sfincs_discharge.nc; set FLOOD_RM_FORCE_WFLOW_REPLAY=1 to rerun Wflow.")
if not RUN_WFLOW_REPLAY:
    print("Wflow replay planning only; configure WFLOW_BIN/wflow.run.command or set FLOOD_RM_RUN_WFLOW_REPLAY=1:")
    for _, row in replay_report.iterrows():
        print(f"\n# {row['submodel_id']}")
        print("  update:", row.get("resolved_update_command") or row["update_command"])
        if row.get("hydromt_runner_issue"):
            print("  hydromt:", row["hydromt_runner_issue"])
        print("  run   :", row["run_command"])

discharge_ready = discharge_nc.exists()
print("\nsfincs_discharge.nc present:", discharge_ready, "->", discharge_nc)


## 3 - Stage SFINCS, apply pluvial rainfall + Wflow discharge, and run

`stage_inland_coupled_scenarios` copies the selected built SFINCS base domain into the
production scenario tree for this event. The cell then opens each staged domain with
HydroMT-SFINCS, writes direct rainfall (`sfincs_netampr.nc`) from the same event precipitation
used by Wflow, applies the merged Wflow discharge GeoDataset at the model `src` points, and
runs SFINCS when a runner is configured.

Set `FLOOD_RM_RUN_SFINCS=0` to stage and write forcing without launching the SFINCS engine.


In [ ]:
FORCE_SFINCS_STAGE = env_flag("FLOOD_RM_FORCE_SFINCS_STAGE", True)

sfincs_run_cfg = config.get("scenario_run", {}) or {}
sfincs_bin = sfincs_run_cfg.get("sfincs_bin") or os.environ.get(sfincs_run_cfg.get("sfincs_bin_env", "SFINCS_BIN"), "")
sfincs_command_available = command_available(sfincs_bin) or shutil.which("docker") is not None
RUN_SFINCS = env_flag("FLOOD_RM_RUN_SFINCS", sfincs_command_available)

display(pd.Series({
    "sfincs_bin": sfincs_bin or "<docker fallback>",
    "sfincs_command_available": sfincs_command_available,
    "run_sfincs": RUN_SFINCS,
    "force_sfincs_stage": FORCE_SFINCS_STAGE,
}, name="sfincs_run_controls"))

sim = {}
if not discharge_nc.exists():
    missing_discharge_message = f"Missing Wflow discharge forcing: {discharge_nc}."
    if RUN_WFLOW_REPLAY:
        raise FileNotFoundError(missing_discharge_message + " Wflow replay was requested but did not produce it.")
    print(missing_discharge_message + " Wflow replay was in planning mode, so SFINCS is not staged yet.")
else:
    scenario_report = stage_inland_coupled_scenarios(
        config,
        {"location_root": location_root},
        catalog_path=example_plan.catalog_path,
        event_ids=[event_id],
        force=FORCE_SFINCS_STAGE,
    )
    display(scenario_report[["event_id", "sfincs_domain_id", "scenario_status", "run_root", "wflow_discharge_forcing"]])

    with xr.open_dataset(discharge_nc) as _dis:
        discharge_ds = _dis.load()
    t_start = pd.Timestamp(discharge_ds["time"].min().values)
    t_stop = pd.Timestamp(discharge_ds["time"].max().values)

    direct_rainfall_cfg = config.get("inland_coupling", {}).get("direct_rainfall", {}) or {}
    stage_direct_rainfall = bool(direct_rainfall_cfg.get("enabled", False))
    if stage_direct_rainfall and not event_precip_nc.exists():
        raise FileNotFoundError(f"Missing SFINCS direct rainfall source: {event_precip_nc}. Run Wflow meteo staging first.")

    for _, scenario in scenario_report.iterrows():
        domain_id = str(scenario["sfincs_domain_id"])
        run_dir = Path(scenario["run_root"])

        sf = SfincsModel(root=str(run_dir), mode="r+")
        sf.read()
        sf.config.update({
            "tref": t_start.to_pydatetime(),
            "tstart": t_start.to_pydatetime(),
            "tstop": t_stop.to_pydatetime(),
        })

        prepared_precip = None
        netamprfile = ""
        if stage_direct_rainfall:
            prepared_precip = run_dir / "aorc_precip_for_sfincs.nc"
            shutil.copy2(event_precip_nc, prepared_precip)
            sf.data_catalog.from_dict({
                "event_precip": {
                    "uri": str(prepared_precip),
                    "data_type": "RasterDataset",
                    "driver": {"name": "raster_xarray"},
                    "metadata": {"crs": 4326},
                }
            })
            # Clear stale meteo pointers before HydroMT-SFINCS creates this event's netampr file.
            sf.config.set("precipfile", None)
            sf.config.set("netamprfile", None)
            sf.precipitation.create(
                precip="event_precip",
                buffer=float(direct_rainfall_cfg.get("buffer_m", 30000.0)),
                cumulative_input=bool(direct_rainfall_cfg.get("cumulative_input", True)),
                time_label=str(direct_rainfall_cfg.get("time_label", "right")),
                aggregate=False,
            )
            sf.precipitation.write()
            netamprfile = "sfincs_netampr.nc"

        # Pass the opened dataset so HydroMT uses the xarray GeoDataset driver instead of the vector fallback.
        sf.discharge_points.create(geodataset=discharge_ds, merge=False)
        sf.write()

        manifest_path = run_dir / "forcing_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest.update({
            "example_notebook": "02_flood/04/run_example.ipynb",
            "sfincs_run_executed": bool(RUN_SFINCS),
            "wflow_discharge_forcing": str(discharge_nc),
            "wflow_source_variable": config["wflow"].get("handoff", {}).get("source_variable", "river_q"),
            "direct_rainfall_enabled": bool(stage_direct_rainfall),
            "direct_rainfall_source": str(event_precip_nc) if stage_direct_rainfall else "",
            "prepared_precip": str(prepared_precip) if prepared_precip else "",
            "netamprfile": netamprfile,
            "direct_rainfall_note": (
                "SFINCS netampr rainfall staged from the same event precipitation used by Wflow."
                if stage_direct_rainfall else "Direct SFINCS rainfall disabled by config."
            ),
        })
        manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

        result = run_sfincs_model(run_dir, config=config) if RUN_SFINCS else None
        sim[domain_id] = {
            "run_dir": run_dir,
            "result": result,
            "dis": sf.discharge_points.data["dis"],
            "t_start": t_start,
        }
        if RUN_SFINCS:
            print(f"{domain_id}: n_src={sf.discharge_points.nr_points} returncode={result.returncode} "
                  f"map={'ok' if result.map_path.exists() else 'MISSING'}")
        else:
            print(f"{domain_id}: staged forcing written; configure SFINCS_BIN/docker or set FLOOD_RM_RUN_SFINCS=1.")


## 4 · Pre-run forcing QA

`plot_inland_coupled_forcing_qa` audits the discharge handoff and the staged SFINCS infiltration
state (initial soil saturation `seff/smax`, `Ksat`) for each example run.

In [ ]:
for domain_id, s in sim.items():
    plot_inland_coupled_forcing_qa(
        forcing_manifest=s["run_dir"] / "forcing_manifest.json",
        event_id=event_id,
        event_label=f"{event_id} · {domain_id}",
    )

if not sim:
    print("No example runs — nothing to QA.")

## 5 · Flood + discharge animation (mp4)

`plot_inland_flood_animation` shows the static peak-depth summary inline and saves the mp4:
land flood depth over a satellite basemap with a synced inflow-discharge hydrograph.

In [ ]:
for domain_id, s in sim.items():
    discharge_df = s["dis"].transpose("time", "index").to_pandas()
    out_mp4 = plot_inland_flood_animation(
        run_root=s["run_dir"],
        out_dir=s["run_dir"] / "figures",
        event_id=event_id,
        event_label=f"{event_id} · {domain_id}",
        discharge=discharge_df,
        t_start=s["t_start"],
    )
    display(HTML(f"<h4>{event_id} · {domain_id} — flood + discharge animation</h4>"))
    display(Video(str(out_mp4), embed=True))

if not sim:
    print("No example runs — nothing to animate.")

## 6 · Post-run diagnostics

Peak flood depth, the SFINCS gauge hydrograph, and the coupling manifest for each example run.

In [ ]:
for domain_id, s in sim.items():
    plot_inland_coupled_postrun_diagnostics(
        run_root=s["run_dir"],
        event_label=f"{event_id} · {domain_id}",
    )

if not sim:
    print("No example runs — nothing to diagnose.")